### SFOS API - Nested & Parameterized Resources

**Purpose:** Demonstrate Template A in-path placeholder substitution for nested sub-resources and item-level actions using type-safe `Domain StrEnum` constants.

**Covered Endpoints:**
1. `Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS` - Sub-resource GET & PATCH
2. `Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS` - Item Action POST
3. `Authentication.SERVERS_GENERIC_IDP_SSO_BY_ID_OR_NAME_FETCH_GROUPS` - Item Action POST
4. `ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE` - Item Action POST
5. `DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE` - Custom Keyed Item GET

---


#### 1. Setup: Import Libraries and Configure Client


In [ ]:
from common import handle_example_error, print_data, setup_client

from sfos_api import ActiveThreatResponse, Authentication, DenialOfService, Logging, NotFoundError, SFOSApiError

# Initialize the SFOS API Client
client = setup_client()
print("✅ SFOS API client initialized")

#### 2. Read Syslog Server Logs Configuration (Sub-Resource GET)

Using `Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS` with `id_or_name` seamlessly substitutes the server ID into the sub-resource path.


In [ ]:
from common import print_data

from sfos_api import Logging, SFOSApiError

endpoint = Logging.LOG_DESTINATIONS_LOG_VIEWER

try:
    result = client.read(endpoint)

    # 1. Print full response object
    print("Full Response:")
    print_data(result)

    # 2. Extract and print log components
    log_components = result["logs"]["logComponents"]
    print("\nLog Components:")
    print_data(log_components, keys=["name", "displayName", "logType"])

except SFOSApiError as e:
    # Error fields are on the exception instance, not on result
    print("Error:", e.error)  # e.g. "badRequest", "notAuthorized"
    print("Message:", e.message)  # Human-readable message
    print("Code:", e.code)  # Status or error code
    print("Status Code:", e.status_code)  # HTTP status code (e.g. 400, 401, 404)
    print("Response Body:", e.response_body)  # Full parsed response dictionary

In [ ]:
endpoint = Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS
server_name = "firewall"

try:
    # Read sub-resource using Endpoint constant
    response = client.read(endpoint, id_or_name=server_name)
    print(f"✅ Fetched log component settings for '{server_name}':")
    print_data(response)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Syslog server '{server_name}' does not exist on the firewall.")
    print("   Guidance: Ensure a syslog server is created under /logging/log-destinations/syslog-servers first.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 3. Update Syslog Server Logs Configuration (Sub-Resource PATCH)

Update log components using the typed Endpoint constant.


In [ ]:
payload = {
    "systemLogs": True,
    "firewallRuleLogs": True,
}

try:
    # Update sub-resource using Endpoint constant
    response = client.update(endpoint, id_or_name=server_name, data=payload)
    print(f"✅ Updated log components for '{server_name}':")
    print_data(response)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Cannot update logs; syslog server '{server_name}' does not exist.")
    print("   Guidance: Ensure the target syslog server is created before updating its log components.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 4. Fetch Active Directory Groups (Item-Level Action POST)

Using `Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS` with `id_or_name`.


In [ ]:
ad_endpoint = Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS
ad_server_name = "AD"

action_payload = {
    "baseDn": "dc=sfos,dc=local",
}

try:
    # Execute action via Endpoint constant with id_or_name
    result = client.create(ad_endpoint, id_or_name=ad_server_name, data=action_payload)
    print(f"✅ Fetched AD groups for '{ad_server_name}':")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Active Directory server '{ad_server_name}' does not exist on the firewall.")
    print("   Guidance: Configure the Active Directory server under /authentication/servers/active-directory first.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 5. Synchronize Third-Party Threat Feed (Item Action POST)
Using `ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE`.


In [ ]:
sync_endpoint = ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE
feed_name = "Feodor"

try:
    # Trigger threat feed sync using Endpoint constant
    result = client.create(sync_endpoint, id_or_name=feed_name)
    print(f"✅ Threat feed '{feed_name}' synchronized:")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Third-party threat feed '{feed_name}' does not exist.")
    print(
        "   Guidance: Add a threat feed under /active-threat-response/thirdparty-threat-feeds before triggering sync."
    )
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 6. Read Denial-of-Service Attack Statistics (Custom Keyed Item GET)

Using `DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE`.


In [ ]:
dos_endpoint = DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE
attack_type = "SYN"  # Value must be 'SYN', 'UDP', 'ICMP', or 'IP'.

try:
    # Read attack statistics for specific flood type
    result = client.read(dos_endpoint, id_or_name=attack_type)
    print(f"✅ DoS attack statistics for '{attack_type}':")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Attack type '{attack_type}' was not recognized by the firewall.")
    print("   Guidance: Valid flood attack types include 'syn-flood', 'udp-flood', 'tcp-flood', etc.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")